In [1]:
import os
import sys
os.chdir('..')
# sys.path.insert('.')

In [2]:
import argparse
import os
import time
import torch
import pandas as pd
from importlib import reload
import json
import numpy as np
from dotenv import load_dotenv
from copy import deepcopy
import glob
import re
import ast
import json
from ast import literal_eval
load_dotenv()

True

In [3]:
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [4]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
import pickle
# Automatically select device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
def load_finetuned_model_lens_from_dir(dir: str, device: str = device) -> HookedTransformer:
    """
    Load a fine-tuned TransformerLens model from a specified directory.

    Args:
        dir (str): Directory containing the model files.
        device (str): Device to load the model onto.
    
    Returns:
        HookedTransformer: The loaded TransformerLens model.
    """
    with open(os.path.join(dir, 'model_config.pkl'), 'rb') as f:
        new_cfg_dict = pickle.load(f)
    new_cfg = HookedTransformerConfig.from_dict(new_cfg_dict)
    new_model = HookedTransformer(new_cfg)
    new_model.load_state_dict(torch.load(os.path.join(dir, 'model.pt'), map_location=device))
    return new_model

/raid/home/m13521157/absa-eap-ig/enveap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Get Counterfacts from Test Data

In [5]:
# List all files in a directory recursively, but stop at the last folder before a file
def list_files_recursively(directory):
	file_list = []
	for root, dirs, files in os.walk(directory):
		for file in files:
			file_list.append(os.path.join(root, file))
	return file_list

In [6]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

def parse_absa_string_gas(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "(tempatnya, bagus, positive); (kolam renangnya, bersih, positive)" becomes:
	[{'A': 'tempatnya', 'S': 'positive', 'O': 'bagus'},
	{'A': 'kolam renangnya', 'S': 'positive', 'O': 'bersih'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	triplets_raw = text.split(';')
	triplets_raw = [i.strip() for i in triplets_raw]
	triplets = []
	for triplet in triplets_raw:
		triplet = triplet.replace('(', '').replace(')', '')
		parts = [part.strip() for part in triplet.split('|')]
		if len(parts) == 3:
			triplet_dict = {'A': parts[0], 'O': parts[1], 'S': parts[2]}
			triplets.append(triplet_dict)
	return triplets
    

In [7]:
parse_absa_string_gas('(kolam renangnya | bersih | positive) ; (tempatnya | bagus | positive)')

[{'A': 'kolam renangnya', 'O': 'bersih', 'S': 'positive'},
 {'A': 'tempatnya', 'O': 'bagus', 'S': 'positive'}]

### Get the candidates

#### Load dataset

In [8]:
lang = 'indo'
dataset_folder = 'corrected_splitopinion_typocorrected'
seed = 123
langs = ['indo']
counterfact_id = 'counterfactsv3.7.4'

In [9]:
model_folder = 'modelsbest_adamw'
files = list_files_recursively(f'outputs/{model_folder}/eap/{dataset_folder}')
models = [os.path.dirname(f) for f in files if 'topk' not in f]
# models = [os.path.dirname(f) for f in files if '2025-10-29 18' in f or '2025-10-29 19' in f]
# models = [f for f in models if not f.endswith('full_sft')]
models = list(set(models))
models

['outputs/modelsbest_adamw/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/full_sft/2025-11-01 12:33:07.824275_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/modelsbest_adamw/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_9584/aos_sequence_variants/full_sft/2025-11-01 12:33:07.008290_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/modelsbest_adamw/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_777/aos_sequence_variants/full_sft/2025-11-01 12:33:08.574211_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/modelsbest_adamw/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_123/aos_sequence_variants/full_sft/2025-11-01 12:33:08.398049_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.

In [10]:
dataset_dict = {}
for lang in langs:
    with open(f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_train_augmented_noreasoning.json', 'r') as f:
        dataset_dict[lang] = json.load(f)

#### Get valid candidates (n number of triples with no null aspects)

In [11]:
valid_candidate = {}
if 'aos' in dataset_folder or 'gas' in dataset_folder:
	step = 1
else:
	step = 5
for lang in langs:
	valid_candidate[lang] = []
	for idx in range(0, len(dataset_dict[lang]), step):
		if 'gas' in dataset_folder:
			num_of_targets = re.findall(r';', dataset_dict[lang][idx]['target'])
		else:
			num_of_targets = re.findall(r'\[SSEP\]', dataset_dict[lang][idx]['target'])

		# Initialize the number of triplets deemed valid
		valid_num_of_targets = [0] # List of valid number of targets
		if len(num_of_targets) in valid_num_of_targets and 'null' not in dataset_dict[lang][idx]['target']:
			valid_candidate[lang].append(dataset_dict[lang][idx])

len(valid_candidate['indo'])

191

#### Get A+O ratio distribution based on data (n number of triplets, no null aspects, less replacement)

In [ ]:
model = HookedTransformer.from_pretrained('Qwen/Qwen2.5-0.5B', device=device)

In [ ]:
valid_candidate = {}
if 'aos' in dataset_folder or 'gas' in dataset_folder:
	step = 1
else:
	step = 5
for lang in langs:
	valid_candidate[lang] = []
	for idx in range(0, len(dataset_dict[lang]), step):
		if 'gas' in dataset_folder:
			num_of_targets = re.findall(r';', dataset_dict[lang][idx]['target'])
		else:
			num_of_targets = re.findall(r'\[SSEP\]', dataset_dict[lang][idx]['target'])

		# Initialize the number of triplets deemed valid
		valid_num_of_targets = [0] # List of valid number of targets
		if len(num_of_targets) in valid_num_of_targets and 'null' not in dataset_dict[lang][idx]['target']:
			valid_candidate[lang].append(dataset_dict[lang][idx])

len(valid_candidate['indo'])

In [ ]:
df_all = pd.DataFrame([i for idx, i in enumerate(dataset_dict['indo']) if idx % step == 0])
df_all

In [ ]:
def return_df_with_token_distribution(df):
	df['input'] = df['input'].apply(lambda x: x.replace('[A] [O] [S]', '').replace('=>', '').strip())

	# Remove punctuation and double spaces in the input
	df['input'] = df['input'].str.replace('[^\w\s]', '', regex=True)
	df['input'] = df['input'].str.replace('\s+', ' ', regex=True)

	# Remove punctuation and double spaces in the target (except for '[', ']')
	df['target'] = df['target'].str.replace('[^\w\s\[\]]', '', regex=True)
	df['target'] = df['target'].str.replace('\s+', ' ', regex=True)

	df['input_tokens_count'] = df['input'].apply(lambda x: len(model.to_str_tokens(x.strip())))
	if 'gas' in dataset_folder:
		df['target_parsed'] = df['target'].apply(lambda x: parse_absa_string_gas(x))
	else:
		df['target_parsed'] = df['target'].apply(lambda x: parse_absa_string(x))

	def count_aspect_and_opinion_tokens(list_of_triplets):
		count = 0
		for triplet in list_of_triplets:
			if triplet['A'] in triplet['O']: # Count only the tokens length of O part if A in O
				count += len(model.to_str_tokens(f" {triplet['O']}"))
			elif triplet['A'] == 'null':
				count += len(model.to_str_tokens(f" {triplet['O']}"))
			else:
				count += len(model.to_str_tokens(f" {triplet['A']}")) + len(model.to_str_tokens(f" {triplet['O']}"))
		return count
	df['ao_tokens_count'] = df['target_parsed'].apply(lambda x: count_aspect_and_opinion_tokens(x))
	df['ratio_ao_to_input'] = df['ao_tokens_count'] / df['input_tokens_count']
	return df

df_all = return_df_with_token_distribution(df_all)

In [ ]:
df_all

In [ ]:
# Plot histogram of ratio_ao_to_input
import matplotlib.pyplot as plt
plt.hist(df_all['ratio_ao_to_input'], bins=20)
plt.xlabel('Ratio of A+O tokens to Input tokens')
plt.ylabel('Frequency')
plt.title('Histogram of Ratio of A+O tokens to Input tokens')
plt.show()

In [ ]:
# Plot cumulative distribution of ratio_ao_to_input (the y label should be the true frequency)
plt.hist(df_all['ratio_ao_to_input'], bins=20, cumulative=True, density=False)
plt.xlabel('Ratio of A+O tokens to Input tokens')
plt.ylabel('Cumulative Frequency')
plt.title('Cumulative Distribution of Ratio of A+O tokens to Input tokens')
plt.show()

#### Output the final dataset

In [12]:
for lang in langs:
    data_for_csv = {
        'index': [instance['sentence_id'] for instance in valid_candidate[lang]],
        'original_pair': [f"{instance['input']} {instance['target']}" for instance in valid_candidate[lang]],
	}
    df_empty_counterfact = pd.DataFrame(data_for_csv)
    # df_empty_counterfact['corrupted_pair'] = np.nan # Placeholder for corrupted pairs
    df_empty_counterfact['corrupted_pair'] = df_empty_counterfact['original_pair'] # Run this if you want to fill in corrupted pairs later (e.g., filtering first and then filling the corrupted pairs)
    os.makedirs(f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}', exist_ok=True)
    df_empty_counterfact.to_csv(f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}/{lang}_counterfacts.csv', index=False)

### Inference

In [13]:
import re
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

In [14]:
# Parse this string:
# "(tempat, strategis, positive); (tempat, dengan pusat keramaian, negative); (parkiran, tebatas, negative)"
# To this list of tuples:
# [('tempat', 'strategis', 'positive'), ('tempat', 'dengan pusat keramaian', 'negative'), ('parkiran', 'tebatas', 'negative')]
def parse_gas_string(text: str):
	"""
	Parse a string containing GAS tuples into a list of tuples.
	Args:
		text (str): Input string containing GAS tuples in the format "(A, O, S); (A, O, S); ...".
	Returns:
		List[Tuple[str, str, str]]: List of tuples where each tuple is (A, O, S).
	"""
	pattern = r"\(([^)]+)\)"
	matches = re.findall(pattern, text)
	result = []
	for match in matches:
		parts = [part.strip() for part in match.split(',')]
		if len(parts) == 3:
			result.append((parts[0], parts[1], parts[2]))
	return result

def extract_triplet_fixed(text):
	try:
		matches = list(re.finditer(r"\[([AOS])\]", text))
		if len(matches) >= 3:
			a_start = matches[0].end()
			o_start = matches[1].end()
			s_start = matches[2].end()
			aspect = text[a_start:matches[1].start()].strip()
			opinion = text[o_start:matches[2].start()].strip()
			sentiment = text[s_start:].split()[0].strip()
			return (aspect, opinion, sentiment)
	except:
		return None
		


In [15]:
parse_absa_string_gas(add_space_around_punctuation('(tempat | strategis | positive); (tempat | dengan pusat keramaian | negative); (parkiran | tebatas | negative)'))

[{'A': 'tempat', 'O': 'strategis', 'S': 'positive'},
 {'A': 'tempat', 'O': 'dengan pusat keramaian', 'S': 'negative'},
 {'A': 'parkiran', 'O': 'tebatas', 'S': 'negative'}]

In [16]:
parse_gas_string(add_space_around_punctuation('(tempat, strategis, positive); (tempat, dengan pusat keramaian, negative); (parkiran, tebatas, negative)'))

[('tempat', 'strategis', 'positive'),
 ('tempat', 'dengan pusat keramaian', 'negative'),
 ('parkiran', 'tebatas', 'negative')]

In [17]:
import re
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

In [18]:
from typing import Optional
def format_counterfactuals(input_path):
	df = pd.read_csv(input_path, encoding="utf-8", quoting=1)

	# df = df.dropna(subset=["corrupted_pair"])
	df['original_sentence'] = df['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip())
	temp_column = df['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[-1].replace('=>', '').strip())
	temp_column = temp_column.apply(lambda x: x.split('[SSEP]')).apply(lambda x: [i.strip() for i in x])
	temp_column = temp_column.apply(lambda x: [extract_triplet_fixed(i) for i in x])
	df['original_triplet'] = deepcopy(temp_column)

	try:
		df['counterfact4_replaced'] = df['corrupted_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip())
		temp_column = df['corrupted_pair'].apply(lambda x: x.split('[A] [O] [S]')[-1].replace('=>', '').strip())
		temp_column = temp_column.apply(lambda x: x.split('[SSEP]'))
		temp_column = temp_column.apply(lambda x: [i.strip() for i in x])
		temp_column = temp_column.apply(lambda x: [extract_triplet_fixed(i) for i in x])
		df['counterfact_triplet4_replaced'] = deepcopy(temp_column)
	except KeyError:
		df['counterfact4_replaced'] = None
		df['counterfact_triplet4_replaced'] = None
		print("KeyError: 'corrupted_pair' is not in a valid format (must be string and no None value). Skipping replacement.")

	df_out = df[['index', 'original_sentence', 'original_triplet', 'counterfact4_replaced', 'counterfact_triplet4_replaced']].copy()
	folder = os.path.dirname(input_path)
	filename = os.path.basename(input_path)
	print(f"Saving formatted data to {os.path.join(folder, f'formatted_{filename}')}")
	df_out.to_csv(os.path.join(folder, f"formatted_{filename}"), index=False)
	print(f"Saved {len(df_out)} rows to {os.path.join(folder, f'formatted_{filename}')}")
	return df_out

def format_counterfactuals_gas(
    input_path: str,
    col_original: str = "original_pair",
    col_counter: str = "corrupted_pair",
    index_col_name: str = "index",
    coerce_index_to_int: bool = True,
) -> pd.DataFrame:
    """
    Returns a DataFrame with:
      - index (preserved or generated), optionally coerced to int64 if safe
      - original_sentence (prompt ending with ' =>')
      - original triplet
      - counterfact        (your code keeps ' =>', retained here)
      - counterfact triplet
    Drops rows where counterfact is NaN/empty.
    """
    pair_re = re.compile(r'^(.*?)\s*=>\s*\((.*?)\)\s*$')

    def parse_pair(value: Optional[str]):
        if pd.isna(value):
            return "", ""
        s = str(value).strip()
        m = pair_re.match(s)
        if m:
            left = m.group(1).strip()
            inner = m.group(2).strip()
            return left, f"( {inner} )"
        if "=>" in s:
            left, right = s.split("=>", 1)
            left, right = left.strip(), right.strip()
            if not (right.startswith("(") and right.endswith(")")):
                right = f"( {right} )"
            return left, right
        return "" | ""

    df_in = pd.read_csv(input_path)

    if index_col_name in df_in.columns:
        idx_vals = df_in[index_col_name].copy()
    else:
        idx_vals = pd.Series(df_in.index, name=index_col_name)

    if coerce_index_to_int:
        try:
            idx_num = pd.to_numeric(idx_vals, errors="coerce")
            idx_num = idx_num.replace([np.inf, -np.inf], np.nan)
            if idx_num.notna().all():
                idx_vals = idx_num.astype("int64")
        except Exception:
            pass

    orig_sentences, orig_triplets = [], []
    cf_sentences, cf_triplets = [], []

    for _, row in df_in.iterrows():
        o_s, o_t = parse_pair(row.get(col_original, ""))
        c_s, c_t = parse_pair(row.get(col_counter, ""))

        orig_sentences.append((o_s + " =>").strip())
        orig_triplets.append(o_t)
        cf_sentences.append((c_s + " =>").strip() if c_s else c_s)
        cf_triplets.append(c_t)

    df_out = pd.DataFrame({
        index_col_name: idx_vals,
        "original_sentence": orig_sentences,
        "original_triplet": orig_triplets,
        "counterfact": cf_sentences,
        "counterfact_triplet": cf_triplets,
    })

    mask = df_out["counterfact"].notna() & (df_out["counterfact"].astype(str).str.strip() != "")
    df_out = df_out.loc[mask].reset_index(drop=True)

    folder = os.path.dirname(input_path)
    filename = os.path.basename(input_path)
    out_path = os.path.join(folder, f"formatted_{filename}")
    print(f"Saving formatted data to {out_path}")
    df_out.to_csv(out_path, index=False)
    print(f"Saved {len(df_out)} rows to {out_path}")

    return df_out

_GAS_TRIPLET_RE = re.compile(r"\(([^()]*)\)")

def _extract_first_triplet(text: str) -> Optional[str]:
    if not isinstance(text, str):
        return None
    m = _GAS_TRIPLET_RE.search(text)
    if not m:
        return None
    parts = [p.strip() for p in m.group(1).split(",")]
    # return f"({', '.join(parts)})" # Old spacing with comma. TODO: Remove/clean this part
    return f"( {' | '.join(parts)} )"


def _normalize_triplet_str(s: str) -> Optional[str]:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s).strip()
    if s.startswith("(") and s.endswith(")"):
        return _extract_first_triplet(s)
    return _extract_first_triplet(s)


def filter_correct_data_gas(
    model,
    data: pd.DataFrame,
    sentence_col: str = "original_sentence",
    label_col: str = "original_triplet",
    max_tokens: int = 60,
    filter_only_correct: bool = True,
    save_path: Optional[str] = None
) -> pd.DataFrame:
    inputs = data[sentence_col].tolist()
    labels = data[label_col].tolist()

    inferences, originals, match_flags = [], [], []

    for prompt, expected_triplet in zip(inputs, labels):
        base_prompt = str(prompt).rstrip()
        if not base_prompt.endswith("=>"):
            base_prompt = base_prompt + " =>"

        output = model.generate(
            input=base_prompt,
            max_new_tokens=max_tokens,
            stop_at_eos=True,
            do_sample=False,
            return_type="str"
        )

        gen_only = output[len(base_prompt):].lstrip() if output.startswith(base_prompt) else output
        gen_triplet_norm = _extract_first_triplet(gen_only)
        exp_triplet_norm = _normalize_triplet_str(expected_triplet)

        is_match = (gen_triplet_norm is not None) and (exp_triplet_norm is not None) and (gen_triplet_norm == exp_triplet_norm)
        if not is_match:
            print(f"Generated triplet: {gen_triplet_norm}, Expected triplet: {exp_triplet_norm}")

        originals.append(exp_triplet_norm if exp_triplet_norm is not None else str(expected_triplet))
        inferences.append(gen_triplet_norm if gen_triplet_norm is not None else "")

        match_flags.append(is_match)

    df_result = data.copy()
    df_result["original_label"] = originals          
    df_result["inference"] = inferences          
    df_result["is_match"] = match_flags

    total = len(df_result)
    correct = int(df_result["is_match"].sum())
    print(f"Correct: {correct} / {total} ({correct / total:.2%})")

    if filter_only_correct:
        df_result = df_result[df_result["is_match"]].reset_index(drop=True)

    if save_path:
        df_result.to_csv(save_path, index=False)

    return df_result


def _normalize_triplet_str(s: str) -> Optional[str]:
    """Return a normalized '(a, b, c)' triplet string or None if not parseable."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s).strip()
    m = _GAS_TRIPLET_RE.search(s if (s.startswith("(") and s.endswith(")")) else f"({s})")
    if not m:
        return None
    parts = [p.strip() for p in m.group(1).split(",")]
    # return f"({', '.join(parts)})" # Old spacing with comma. TODO: Remove/clean this part
    return f"( {' | '.join(parts)} )"


def convert_triplet_string(triplet_str: str) -> tuple:
	"""
	Safely parses a stringified triplet like '[("aspect", "opinion", "sentiment")]'
	and returns the individual components.

	Returns:
		Tuple of (aspect, opinion, sentiment) or empty strings if invalid.
	"""
	try:
		triplet = ast.literal_eval(triplet_str)
		return tuple(triplet)
	except (ValueError, SyntaxError, IndexError):
		return "", "", ""

def filter_correct_data(model, df, input_col, label_col, filter_mode="aos", filter_only_correct=True, save_path=None, max_tokens=150):

	if filter_mode == "aos":
		suffix = ' [A] [O] [S]'
	elif filter_mode == "aosarrow":
		suffix = ' [A] [O] [S] =>'
	elif filter_mode == "gas":
		suffix = ''
	elif filter_mode == "gasarrow":
		suffix = ' =>'
	else:
		raise ValueError(f"Invalid filter_mode '{filter_mode}'. Must be one of {{'aos', 'gas', 'gasarrow'}}.")
	# For testing, take 5 first and 5 last instances of the df
	# df = pd.concat([df.iloc[:5], df.iloc[-5:]]).reset_index(drop=True)
	inputs = df[input_col].tolist()
	labels = df[label_col].tolist()

	results, expected_labels, match_flags = [], [], []

	for prompt, label_raw in zip(inputs, labels):
		full_prompt = prompt + suffix
		print(f"Processing prompt: {full_prompt}")
		output = model.generate(
			input=full_prompt,
			max_new_tokens=max_tokens,
			stop_at_eos=True,
			do_sample=False,
			return_type="str"
		)

		# Remove special tokens and clean up
		raw_output = re.sub(r"<\|endoftext\|>", "", output)

		# Handle multiple triplets
		is_match = True
		triplets_str = raw_output.split("[A] [O] [S]")[-1].strip()
		triplets_str_temp = triplets_str.split("[SSEP]")
		# print(f"Triplets string: {triplets_str}")
		triplets_str_temp = [i.strip() for i in triplets_str_temp]
		labels = convert_triplet_string(label_raw)
		# print(f'Labels: {labels}')
		for triplet_str in triplets_str_temp:
			triplet = extract_triplet_fixed(triplet_str)
			if triplet not in labels:
				print(f"Mismatch found: {triplet} not in {labels}")
				is_match = False
				break
		
		formatted_labels = [f"[A] {label[0]} [O] {label[1]} [S] {label[2]}" for label in labels]
		formatted_labels = " [SSEP] ".join(formatted_labels)
		if is_match:
			results.append(formatted_labels) # Same ordering as the formatted labels
		else:
			results.append(triplets_str)
		expected_labels.append(formatted_labels)
		match_flags.append(is_match)

	df_result = df.copy()
	df_result["original_label"] = expected_labels
	df_result["inference"] = results
	df_result["is_match"] = match_flags

	total = len(df_result)
	correct = df_result["is_match"].sum()
	print(f"Correct: {correct} / {total} ({correct / total:.2%}) with mode [{filter_mode}]")

	if filter_only_correct:
		df_result = df_result[df_result["is_match"]].reset_index(drop=True)
		
	if save_path:
		df_result.to_csv(save_path, index=False)
	
	return df_result

### Testing per step (only for debugging purposes, skip if not needed)

In [ ]:
test_path = 'hotel_dataset/test_counterfacts/indo_counterfacts.csv'
df_counterfact_test = pd.read_csv(test_path)
format_counterfactuals(test_path)
df_counterfact_test = pd.read_csv(os.path.dirname(test_path) + f"/formatted_{os.path.basename(test_path)}")
df_counterfact_test

In [ ]:
# This is for debugging purpose (only one model not several)
model_path = None
for temp in models:
	if 'indo' in temp:
		model_path = temp
		break
model = load_finetuned_model_lens_from_dir(model_path)
device = (
	torch.device("mps") if torch.backends.mps.is_available()
	else torch.device("cuda") if torch.cuda.is_available()
	else torch.device("cpu")
)
model.to(device)
model.eval()

print(model_path)

In [ ]:
# This is for debugging purpose (only one model not several)
filtered_data_path = f'temp/debug_for_multitriplet.csv'
filtered_df = filter_correct_data(
	model,
	df_counterfact_test,
	"original_sentence",
	"original_triplet",
	filter_mode="AOS",
	filter_only_correct=False,
	save_path=filtered_data_path,
	max_tokens=150  # Adjusted max_tokens to 150 for better performance
)

In [ ]:
filtered_df

In [ ]:
false_instance = filtered_df.loc[filtered_df['index'] == 2900, :]
target = false_instance['original_label'].values[0]
pred = false_instance['inference'].values[0]
print(f"Target: {target}")
print(f"Prediction: {pred}")
print(f"Match: {target == pred}")

### Filter for all models

In [19]:
filtered_dfs = {}
for model_path in models:
	model = load_finetuned_model_lens_from_dir(model_path)
	device = (
		torch.device("mps") if torch.backends.mps.is_available()
		else torch.device("cuda") if torch.cuda.is_available()
		else torch.device("cpu")
	)
	model.to(device)
	model.eval()

	print(model_path)

	if 'indo' in model_path:
		dataset_path = f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}/indo_counterfacts.csv'
		dataset_path = f'hotel_dataset/counterfactsv3.5/corrected_splitopinion_typocorrected/indo_counterfacts.csv'
		language = 'indo'
	elif 'eng' in model_path:
		dataset_path = f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}/eng_counterfacts.csv'
		language = 'eng'
	elif 'sunda' in model_path:
		dataset_path = f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}/sunda_counterfacts.csv'
		language = 'sunda'
	else:
		raise ValueError("Unknown model language in path: " + model_path)
	
	df = pd.read_csv(dataset_path)
	print(f"Dataset loaded from {dataset_path} ({len(df)} rows)")
	if "original_pair" in df.columns:
		if 'gas' in dataset_folder:
			format_counterfactuals_gas(dataset_path)
		else:
			format_counterfactuals(dataset_path)
		folder = os.path.dirname(dataset_path)
		filename = os.path.basename(dataset_path)
		formated_path = os.path.join(folder, f"formatted_{filename}")
		df = pd.read_csv(formated_path)
	print(f"Columns in dataframe: {df.columns.tolist()}")
	seed = model_path.split('/')[5]
	filtered_data_path = f'temp/{model_folder}/{dataset_folder}/empty_{counterfact_id}/{language}_{seed}.csv'
	os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
	id = filtered_data_path.split('/')[-1]
	if 'gas' in dataset_folder:
		filtered_df = filter_correct_data_gas(
			model,
			df,
			"original_sentence",
			"original_triplet",
			filter_only_correct=True,
			save_path=filtered_data_path,
			max_tokens=150  # Adjusted max_tokens to 150 for better performance
		)
	else:
		filtered_df = filter_correct_data(
			model,
			df,	
			"original_sentence",
			"original_triplet",
			filter_mode="aos",
			filter_only_correct=True,
			save_path=filtered_data_path,
			max_tokens=150  # Adjusted max_tokens to 150 for better performance
		)
	filtered_dfs[id] = filtered_df.copy()

Moving model to device:  cuda
outputs/modelsbest_adamw/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/full_sft/2025-11-01 12:33:07.824275_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20
Dataset loaded from hotel_dataset/counterfactsv3.5/corrected_splitopinion_typocorrected/indo_counterfacts.csv (100 rows)
Saving formatted data to hotel_dataset/counterfactsv3.5/corrected_splitopinion_typocorrected/formatted_indo_counterfacts.csv
Saved 100 rows to hotel_dataset/counterfactsv3.5/corrected_splitopinion_typocorrected/formatted_indo_counterfacts.csv
Columns in dataframe: ['index', 'original_sentence', 'original_triplet', 'counterfact4_replaced', 'counterfact_triplet4_replaced']
Processing prompt: tidak dapat snack . setelah di keluhan , baru dikasik snacknya . [A] [O] [S]


  9%|▊         | 13/150 [00:01<00:12, 10.66it/s]


Processing prompt: kamarnya oke . [A] [O] [S]


 10%|█         | 15/150 [00:00<00:04, 29.32it/s]


Processing prompt: tempat tdr kurang bersih . [A] [O] [S]


...

Processing prompt: karena terlalu banyak kamar jadi tidak diperhatikan masalah kebersihan . [A] [O] [S]


 18%|█▊        | 27/150 [00:00<00:04, 29.76it/s]


Processing prompt: tidak ada wifi sesuai yang di iklankan . [A] [O] [S]


  9%|▊         | 13/150 [00:00<00:04, 28.95it/s]

Correct: 100 / 100 (100.00%) with mode [aos]


In [20]:
filtered_dfs = {}
parent_result_path = f'temp/{model_folder}/{dataset_folder}/empty_{counterfact_id}'
# parent_result_path2 = f'temp/{model_folder}/corrected_splitopinion_typocorrected_aos/empty_counterfactsv3.7.2'
results_path = os.listdir(parent_result_path)
print('Taking results from:', parent_result_path)
for path in results_path:
    filtered_dfs[path] = pd.read_csv(os.path.join(parent_result_path, path))
    # filtered_dfs[path + '_2'] = pd.read_csv(os.path.join(parent_result_path2, path))
print(filtered_dfs.keys())
print(len(filtered_dfs))

Taking results from: temp/modelsbest_adamw/corrected_splitopinion_typocorrected/empty_counterfactsv3.7.4
dict_keys(['indo_seed_9584.csv', 'indo_seed_123.csv', 'indo_seed_31415.csv', 'indo_seed_2024.csv', 'indo_seed_777.csv'])
5


In [21]:
indexes = set()
first = True
for df in filtered_dfs.values():
	if first:
		indexes = set(df['index'].tolist())
		first = False
	else:
		# Get the intersection of indexes
		indexes.intersection_update(df['index'].tolist())

# Convert to list and sort
indexes = sorted(list(indexes))
len(indexes)

97

### Get the data with the valid indexes

In [22]:
df_check = pd.read_csv(f'hotel_dataset/counterfactsv3.5/corrected_splitopinion_typocorrected/indo_counterfacts.csv')
# df_check = pd.read_csv(f'hotel_dataset/counterfactsv3.2_aosonly/corrected_splitopinion_typocorrected_aos/indo_counterfacts.csv')
df_check

,index,original_pair,corrupted_pair
0,4,"tidak dapat snack . setelah di keluhan , baru ...","baik sangat radio . setelah di keluhan , baru ..."
1,10,kamarnya oke . [A] [O] [S] [A] kamarnya [O] ok...,pot bunga busuk . [A] [O] [S] [A] pot bunga [...
2,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,baju anakku cukup cepat . [A] [O] [S] [A] baju...
3,50,tetapi sayangnya di kamar yang saya tempati ti...,tetapi sayangnya di kamar yang saya tempati ce...
4,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,baik sangat lilin . [A] [O] [S] [A] lilin [O] ...
...,...,...,...
95,2399,"terimakasih airy , layanannya sangat sangat ba...","erimakasih airy , peraturan pahit, moga kedepa..."
96,2409,tidak ada lift . kesulitannya hanya mengangkut...,baik amat radio . kesulitannya hanya mengangku...
97,2415,kamar bersih hotel berada disamping gran mall ...,angin kotor hotel berada disamping gran mall b...
98,2458,karena terlalu banyak kamar jadi tidak diperha...,"tabil sekali , jadi proses berjalan sangat mul..."


In [23]:
df_check['false_indexes'] = df_check['index'].isin(indexes)
df_check[~df_check['false_indexes']]

,index,original_pair,corrupted_pair,false_indexes
9,134,"ketika kami menginap , pelayanannya ramah . [A...","ketika kami menginap , kucing hitam mahal . [A...",False
15,219,alhamdulillah ya kamar lumayan luas . [A] [O] ...,alhamdulillah ya meja sangat pahit . [A] [O] [...,False
37,842,bagus semua kok . [A] [O] [S] [A] semua [O] ba...,macet radio kok . [A] [O] [S] [A] radio [O] ma...,False


In [24]:
final_df = df_check[df_check['false_indexes']]
final_df['num_of_triplets'] = final_df['original_pair'].apply(lambda x: len(re.findall(r'\[SSEP\]', x)) + 1)
final_df

/tmp/ipykernel_2011140/4016019287.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['num_of_triplets'] = final_df['original_pair'].apply(lambda x: len(re.findall(r'\[SSEP\]', x)) + 1)


,index,original_pair,corrupted_pair,false_indexes,num_of_triplets
0,4,"tidak dapat snack . setelah di keluhan , baru ...","baik sangat radio . setelah di keluhan , baru ...",True,1
1,10,kamarnya oke . [A] [O] [S] [A] kamarnya [O] ok...,pot bunga busuk . [A] [O] [S] [A] pot bunga [...,True,1
2,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,baju anakku cukup cepat . [A] [O] [S] [A] baju...,True,1
3,50,tetapi sayangnya di kamar yang saya tempati ti...,tetapi sayangnya di kamar yang saya tempati ce...,True,1
4,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,baik sangat lilin . [A] [O] [S] [A] lilin [O] ...,True,1
...,...,...,...,...,...
95,2399,"terimakasih airy , layanannya sangat sangat ba...","erimakasih airy , peraturan pahit, moga kedepa...",True,1
96,2409,tidak ada lift . kesulitannya hanya mengangkut...,baik amat radio . kesulitannya hanya mengangku...,True,1
97,2415,kamar bersih hotel berada disamping gran mall ...,angin kotor hotel berada disamping gran mall b...,True,1
98,2458,karena terlalu banyak kamar jadi tidak diperha...,"tabil sekali , jadi proses berjalan sangat mul...",True,1


In [25]:
dataset_folder = 'corrected_splitopinion_typocorrected'

In [26]:
os.makedirs(f'hotel_dataset/{counterfact_id}/{dataset_folder}', exist_ok=True)
final_df.to_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts.csv', index=False)
print(f"Saved {len(final_df)} rows to hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts.csv")

Saved 97 rows to hotel_dataset/counterfactsv3.7.4/corrected_splitopinion_typocorrected/indo_counterfacts.csv


In [ ]:
counterfact_id = 'counterfactsv3.4.1'
dataset_folder = 'corrected_splitopinion_typocorrected'

In [ ]:
# From here, you have to use MvP format, gas format is only supported until getting the filtered indexes
original_df = pd.read_csv(f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}/{lang}_counterfacts.csv')
original_df = original_df[original_df['index'].isin(indexes)].reset_index(drop=True)
original_df['num_of_triplets'] = original_df['original_pair'].apply(lambda x: len(re.findall(r'\[SSEP\]', x)) + 1)
original_df

In [ ]:
original_df['input'] = original_df['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip() + ' [A] [O] [S]')
original_df['target'] = original_df['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[-1].strip())
original_df

In [ ]:
original_df_dist = return_df_with_token_distribution(original_df.rename({'original_pair': 'input', 'target': 'original_triplet'}).copy())
original_df_dist

In [ ]:
original_df_dist.sort_values(by=['ratio_ao_to_input'], ascending=False)

In [ ]:
new_df = original_df_dist.loc[original_df_dist['ratio_ao_to_input'] <= 2, :].copy()

In [ ]:
new_df['corrupted_pair'] = np.nan

In [ ]:
def move_from_check(row):
	if row['index'] in df_check['index'].values:
		return df_check.loc[df_check['index'] == row['index'], 'corrupted_pair'].values[0]
	return np.nan

In [ ]:
new_df['corrupted_pair'] = new_df.apply(lambda row: move_from_check(row), axis=1)

In [ ]:
new_df

In [ ]:
new_df[['index', 'original_pair', 'corrupted_pair', 'ratio_ao_to_input']].to_csv(f'hotel_dataset/empty_{counterfact_id}/{dataset_folder}/indo_counterfacts_tobefilledv2.csv', index=False)

In [ ]:
# Clip values greater than 1 to 1
true_distribution = df_all['ratio_ao_to_input'].copy()
true_distribution[true_distribution > 1] = 1
true_distribution

In [ ]:
# Take the value of true_distribution, make the value_counts, and then make binning to 20 bins from 0 to 1
binned_true_distribution = pd.cut(true_distribution, bins=np.linspace(0, 1, 21), include_lowest=False)
binned_true_distribution.value_counts()

In [ ]:
# Sample original_df_dist to have the same 'ratio_ao_to_input' distribution, take from the binned_true_distribution value counts
# Sample original_df_dist to have the same 'ratio_ao_to_input' distribution, take from the binned_true_distribution value counts
sampled_indexes = []
true_count = binned_true_distribution.value_counts().sort_index()
for bin_range, count in true_count.items():
	bin_df = original_df_dist[(original_df_dist['ratio_ao_to_input'] > bin_range.left) & (original_df_dist['ratio_ao_to_input'] <= bin_range.right)]
	if len(bin_df) == 0:
		continue
	sampled_bin_df = bin_df.sample(n=min(count, len(bin_df)), random_state=42)
	sampled_indexes.extend(sampled_bin_df.index.tolist())

In [ ]:
len(sampled_indexes)

In [ ]:
original_df.to_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts_tobefilled.csv', index=False)

In [ ]:
# Select 50 rows randomly with seed from original_df, returning a new dataframe
sampled_df = original_df.sample(n=75, random_state=42).sort_values(by='index')
sampled_df

In [ ]:
# Save to CSV
sampled_df.to_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts_tobefilled_sampled.csv', index=False)